# PDF Chunking — pypdf

Reads the extracted text cached by `PDF Parsing (pypdf)` in `rag_chatbot_dev.bronze.doc_parsed_pypdf`, splits it into overlapping chunks, and upserts the chunks into the `rag_chatbot_dev.silver.doc_chunks_pypdf` Delta table for downstream RAG use and side-by-side experimentation with the built-in functions pipeline (`doc_chunks_builtin`).

In [0]:
# --- Configuration -------------------------------------------------------
CATALOG = "rag_chatbot_dev"
TEXT_TABLE = f"{CATALOG}.bronze.doc_parsed_pypdf"       # raw text cache written by PDF Parsing (pypdf)
CHUNK_TABLE = f"{CATALOG}.silver.doc_chunks_pypdf"     # target Delta table (pypdf pipeline, silver layer)
CHUNK_SIZE = 1000      # characters per chunk
CHUNK_OVERLAP = 200    # character overlap between consecutive chunks

from pyspark.sql import functions as F

# Text extraction is decoupled: read the cached text instead of re-parsing PDFs
texts_df = spark.table(TEXT_TABLE)
print(f"Loaded {texts_df.count()} document text(s) from {TEXT_TABLE}")
display(texts_df.select("doc_name", F.length("text").alias("text_chars")))

Loaded 4 document text(s) from rag_chatbot_dev.bronze.doc_parsed_pypdf


doc_name,text_chars
expense_policy.pdf,919
hr_leave_policy.pdf,1021
incident_runbook.pdf,1118
product_faq.pdf,1064


In [0]:
import re

import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.functions import pandas_udf


@pandas_udf("array<string>")
def chunk_text(texts: pd.Series) -> pd.Series:
    """Split one document's text into overlapping chunks of fixed size."""
    chunks_out = []
    for text in texts:
        # Collapse whitespace so chunks are clean, contiguous spans of text
        cleaned = re.sub(r"\s+", " ", text or "").strip()

        chunks = []
        start = 0
        while start < len(cleaned):
            chunks.append(cleaned[start : start + CHUNK_SIZE])
            if start + CHUNK_SIZE >= len(cleaned):
                break
            start += CHUNK_SIZE - CHUNK_OVERLAP
        chunks_out.append(chunks)
    return pd.Series(chunks_out)


# Chunk the cached texts, then attach stable ids and metadata
chunks_df = (
    texts_df
    .withColumn("chunks", chunk_text(F.col("text")))
    .select("doc_path", "doc_name", "chunks")
    .select("doc_path", "doc_name", F.posexplode("chunks").alias("chunk_num", "content"))
    .withColumn("chunk_id", F.md5(F.concat_ws("_", "doc_path", "chunk_num")))
    .withColumn("num_chars", F.length("content"))
    .withColumn("ingested_at", F.current_timestamp())
)

num_chunks = chunks_df.count()
num_docs = chunks_df.select("doc_path").distinct().count()
print(f"Produced {num_chunks} chunk(s) from {num_docs} document(s) with text content")

Produced 7 chunk(s) from 4 document(s) with text content


In [0]:
# Target table: primary key + change data feed so it is ready to back a
# Vector Search Delta Sync index later.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CHUNK_TABLE} (
    chunk_id     STRING NOT NULL,
    doc_path     STRING NOT NULL,
    doc_name     STRING,
    chunk_num    INT,
    content      STRING,
    num_chars    INT,
    ingested_at  TIMESTAMP,
    CONSTRAINT doc_chunks_pk PRIMARY KEY (chunk_id)
) TBLPROPERTIES (delta.enableChangeDataFeed = true);
""")

# Idempotent upsert: re-running the notebook refreshes chunks instead of
# duplicating them. WHEN NOT MATCHED BY SOURCE deletes target rows that no
# longer exist in the source — trailing chunks of documents that shrank and
# all chunks of documents removed from the volume.
chunks_df.createOrReplaceTempView("new_chunks")

spark.sql(f"""
MERGE INTO {CHUNK_TABLE} AS t
USING new_chunks AS s
ON t.chunk_id = s.chunk_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
WHEN NOT MATCHED BY SOURCE THEN DELETE
""")

# --- Verification --------------------------------------------------------
print(f"Total rows in {CHUNK_TABLE}: {spark.table(CHUNK_TABLE).count()}")
display(
    spark.table(CHUNK_TABLE)
    .groupBy("doc_name")
    .agg(
        F.count("*").alias("chunks"),
        F.min("chunk_num").alias("first_chunk"),
        F.max("chunk_num").alias("last_chunk"),
        F.sum("num_chars").alias("total_chars"),
    )
)
display(spark.table(CHUNK_TABLE).orderBy("doc_name", "chunk_num").limit(5))

Total rows in rag_chatbot_dev.silver.doc_chunks_pypdf: 7


doc_name,chunks,first_chunk,last_chunk,total_chars
product_faq.pdf,2,0,1,1263
incident_runbook.pdf,2,0,1,1317
hr_leave_policy.pdf,2,0,1,1220
expense_policy.pdf,1,0,0,918


chunk_id,doc_path,doc_name,chunk_num,content,num_chars,ingested_at
c3a3de8911ae0699404e47a3b4170727,/Volumes/rag_chatbot_dev/bronze/docs/expense_policy.pdf,expense_policy.pdf,0,"ACME CORP EXPENSE POLICY Scope: This policy applies to all employees who submit business expenses. Travel Booking: Flights must be booked at least 14 days before departure when possible. Economy class is the standard for flights under 6 hours. Business class is allowed for flights over 6 hours with VP approval. Hotels: The nightly limit is 180 dollars in most cities and 250 dollars in London, New York, and Tokyo. Breakfast is reimbursed up to 20 dollars per day. Meals during travel: Up to 60 dollars per day, with receipts required for any single meal above 25 dollars. Submission: Expense reports must be submitted within 30 days of the expense date through the Finance portal. Late submissions over 90 days are not reimbursed. Approval: Expenses under 500 dollars are approved by the direct manager. Expenses between 500 and 5000 dollars require the department head. Expenses above 5000 dollars require the CFO.",918,2026-11-08T14:06:53.265Z
4f1c996a1183318a3543fc62fc8ff4df,/Volumes/rag_chatbot_dev/bronze/docs/hr_leave_policy.pdf,hr_leave_policy.pdf,0,Acme Corp Leave Policy (Version 3.2) Annual Leave All full-time employees receive 24 days of paid annual leave per calendar year. Leave accrues monthly at 2 days per month. Unused leave up to 5 days may be carried over to the next year. Any balance beyond 5 days expires on March 31. Sick Leave Employees receive 12 days of paid sick leave per year. A doctor's note is required for sick leave longer than 3 consecutive days. Sick leave does not carry over. Parental Leave Primary caregivers receive 26 weeks of fully paid parental leave. Secondary caregivers receive 8 weeks of fully paid leave. Leave must begin within 12 months of the child's arrival. How to Request Leave Submit requests through the People Portal at least 10 working days in advance for leave longer than 5 days. Short leave of 1 to 2 days requires manager approval only. Requests are answered within 2 working days. Public Holidays Acme Corp observes 10 public holidays per year. The holiday calendar is published in the People P,1000,2026-11-08T14:06:53.265Z
5cb4aa57a9d0333f066e8fea2fe7ab52,/Volumes/rag_chatbot_dev/bronze/docs/hr_leave_policy.pdf,hr_leave_policy.pdf,1,to 2 days requires manager approval only. Requests are answered within 2 working days. Public Holidays Acme Corp observes 10 public holidays per year. The holiday calendar is published in the People Portal each December.,220,2026-11-08T14:06:53.265Z
301dac1b5a320f41d7b15e366f60705b,/Volumes/rag_chatbot_dev/bronze/docs/incident_runbook.pdf,incident_runbook.pdf,0,"Incident Response Runbook: Payment Service Severity Levels SEV1 means the payment service is fully down or more than 20 percent of transactions are failing. SEV2 means degraded performance with latency above 2 seconds at the 95th percentile. SEV3 covers minor issues with no customer impact. Initial Response (SEV1 and SEV2) 1. Page the on-call engineer through PagerDuty within 5 minutes of detection. 2. Open an incident channel named #inc-YYYYMMDD-short-description in Slack. 3. Assign an Incident Commander. For SEV1, the Incident Commander must be a Staff Engineer or above. 4. Post the first status update on the public status page within 15 minutes. Rollback If the incident started within 60 minutes of a deployment, roll back the deployment using the release pipeline: acme-deploy rollback payment-service --env prod. Rollbacks must be approved by the Incident Commander. Post-Incident Review A post-incident review must be completed within 5 working days for every SEV1 and SEV2 incident. T",1000,2026-11-08T14:06:53.265Z
cfe1d91a2d2c95184e52bceff7d08dda,/Volumes/rag_chatbot_dev/bronze/docs/incident_runbook.pdf,incident_runbook.pdf,1,"ayment-service --env prod. Rollbacks must be approved by the Incident Commander. Post-Incident Review A po